# Fine-tuning Sophia no Colab

Ative um runtime GPU. Execute de cima para baixo e preserve o mesmo diretório no Drive para retomar.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
repo = Path('/content/sophia-36b-colab')
if not repo.exists():
    !git clone https://github.com/walkowicz19/sophia-36b-colab.git /content/sophia-36b-colab
%cd /content/sophia-36b-colab
# Em uma revisão de PR, use a branch ou o SHA do PR. Em retomadas, use o mesmo SHA.
git_ref = 'codex/sophia-colab-reliability'
!git checkout {git_ref}
!git rev-parse HEAD
%pip install -r requirements-colab.txt


Reinicie a sessão se a instalação do Colab pedir. Os comandos abaixo executam em processos Python separados. Para retomar, use a mesma revisão Git e as mesmas dependências.

In [ ]:
%cd /content/sophia-36b-colab
import os
import subprocess
from pathlib import Path

drive_root = Path('/content/drive/MyDrive')
assert drive_root.is_dir(), 'Monte o Google Drive primeiro'
persistent = drive_root / 'sophia-runs' / 'k2-qlora-01'
persistent.mkdir(parents=True, exist_ok=True)
subprocess.run(['python', 'validate_dataset.py'], check=True)
subprocess.run(['python', 'colab_phase0_smoke_test.py', '--steps', '10'], check=True)


In [ ]:
# Preencha somente se quiser publicar o melhor adapter em um repositório HF privado.
hf_repo_id = ''
command = ['python', 'colab_phase1_train.py', '--persistent-dir', str(persistent)]
if hf_repo_id:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    command += ['--hf-repo-id', hf_repo_id]
subprocess.run(command, check=True)


In [ ]:
subprocess.run(['python', 'evaluate_sophia_36b_gauntlet.py', '--identity-only',
                '--report', str(persistent / 'identity_report.json')], check=True)
